# PricePilot Dataset Pipeline & Feature Engineering

This notebook demonstrates the complete data engineering workflow for PricePilot:
1. **Download Raw Data** using the PricePilot Python SDK.
2. **Save Raw Exports** to the versioned local filesystem.
3. **Clean & Preprocess Data** using reusable preprocessors.
4. **Build ML-Ready Features** using feature builders.
5. **Export Feature Sets** for future model training.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from pricepilot import PricePilotClient
from pricepilot_ml.preprocessing import (
    MissingValueHandler,
    DuplicateRemoval,
    OutlierDetector,
    CategoricalEncoder,
    PriceNormalizer,
    FeatureScaler,
    TimestampParser,
    NullSafety
)
from pricepilot_ml.feature_engineering import (
    ProductFeatureBuilder,
    UserFeatureBuilder,
    InteractionFeatureBuilder,
    RecommendationFeatureBuilder
)
from pricepilot_ml.datasets import DatasetExporter

## 1. Initialize Client & Download Raw Datasets

We authenticate and query the REST dataset endpoints using the Python SDK.

In [ ]:
# Initialize Client
client = PricePilotClient(base_url="http://localhost:8080/api/v1")

print("Authenticating...")
try:
    # Standard mock login/auth setup
    client.auth.login(email="admin@example.com", password="admin123")
    print("Authenticated successfully.")
except Exception:
    print("Backend offline. Using mock datasets for demonstration.")

## 2. Load / Mock Raw Datasets

If the backend is not running, we generate mock data to showcase the preprocessing pipeline.

In [ ]:
try:
    df_products = client.datasets.products_dataframe()
    df_analytics = client.datasets.product_analytics_dataframe()
    df_interactions = client.datasets.interaction_events_dataframe()
    df_dashboard = client.datasets.dashboard_summary_dataframe()
except Exception:
    # Generate Mock Data
    df_products = pd.DataFrame({
        "id": ["p1", "p2", "p3", "p4"],
        "name": ["iPhone 15", "Sony WH-1000XM4", "Nike Air Max", "Samsung S23"],
        "brand": ["Apple", "Sony", "Nike", "Samsung"],
        "category": ["Electronics", "Electronics", "Footwear", "Electronics"],
        "description": ["Smartphone", "Headphones", "Running Shoes", "Smartphone"],
        "archived": [False, False, False, True],
        "createdAt": ["2026-07-01T10:00:00", "2026-07-02T12:00:00", "2026-07-03T15:00:00", "2026-07-04T09:00:00"],
        "currentMinPrice": [999.0, 348.0, 120.0, None],
        "originalMinPrice": [999.0, 399.0, 150.0, None],
        "sellerCount": [3, 2, 4, 0],
        "averageSellerRating": [4.8, 4.5, 4.2, None]
    })
    
    df_analytics = pd.DataFrame({
        "productId": ["p1", "p2", "p3"],
        "viewCount": [120, 85, 340],
        "saveCount": [15, 8, 45],
        "watchlistCount": [10, 4, 25],
        "priceChangeCount": [2, 1, 3]
    })
    
    df_interactions = pd.DataFrame({
        "userId": ["u1", "u1", "u2", "u2"],
        "productId": ["p1", "p2", "p3", "p1"],
        "interactionType": ["PRODUCT_VIEW", "PRODUCT_VIEW", "PRODUCT_VIEW", "PRODUCT_SAVE"],
        "createdAt": ["2026-07-05T12:00:00", "2026-07-05T12:15:00", "2026-07-06T14:00:00", "2026-07-06T14:10:00"],
        "metadataJson": ["{}", "{}", "{}", "{}"]
    })
    
    df_dashboard = pd.DataFrame({
        "userId": ["u1", "u2"],
        "email": ["john@example.com", "sarah@example.com"],
        "role": ["USER", "USER"],
        "savedCount": [1, 2],
        "watchlistCount": [2, 3],
        "totalActivitiesCount": [10, 15],
        "activePriceAlertsCount": [1, 2]
    })

print("Products shape:", df_products.shape)
print("Analytics shape:", df_analytics.shape)

## 3. Version & Save Raw Datasets

We save the downloaded raw data in `datasets/raw/` with full SemVer and generation metadata.

In [ ]:
exporter = DatasetExporter(base_dir="..")

csv_p, meta_p = exporter.export(
    df=df_products,
    name="products",
    stage="raw",
    version="1.0.0",
    schema_version="1.0",
    generated_by="example-pipeline",
    future_training_target="None"
)
print(f"Exported raw products to: {csv_p}")

## 4. Data Cleaning Pipeline

We process raw data through missing value imputation, outlier handling, and date parsing.

In [ ]:
# Parse Date
df_products = TimestampParser.parse(df_products, col="createdAt")

# Handle Missing values in products
imputer = MissingValueHandler(numeric_strategy="median", categorical_strategy="constant")
df_products_clean = imputer.fit_transform(
    df_products,
    numeric_cols=["currentMinPrice", "originalMinPrice", "averageSellerRating"],
    categorical_cols=["brand", "category"]
)

# Handle Outliers in price
outlier_det = OutlierDetector(factor=1.5)
df_products_clean = outlier_det.fit_transform(df_products_clean, cols=["currentMinPrice"], strategy="cap")

print("Cleaned Products prices:\n", df_products_clean[["name", "currentMinPrice"]])

## 5. Feature Engineering

We generate high-quality ML features for products, users, interactions, and recommendation pairs.

In [ ]:
# 1. Product features
prod_builder = ProductFeatureBuilder()
df_product_features = prod_builder.build_features(df_products_clean, df_analytics)
print("Product Features sample:\n", df_product_features[["name", "viewCount", "trendingScore", "discount_ratio"]].head())

# 2. User features
user_builder = UserFeatureBuilder()
df_user_features = user_builder.build_features(df_dashboard, df_interactions, df_products_clean)
print("\nUser Features sample:\n", df_user_features[["userId", "preferredCategory", "averagePriceViewed"]].head())

# 3. Recommendation candidates (Cross-join user-product pairs)
rec_builder = RecommendationFeatureBuilder()
df_recommendation_features = rec_builder.build_features(df_user_features, df_product_features)
print("\nRecommendation candidate pairs shape:", df_recommendation_features.shape)
print("Recommendation Features sample:\n", df_recommendation_features[["userId", "productId", "category_match", "brand_match"]].head())

## 6. Export Feature Sets for Training

The final engineered feature set is saved in `datasets/feature_sets/` with a version schema ready for model training.

In [ ]:
csv_feat, meta_feat = exporter.export(
    df=df_recommendation_features,
    name="recommendations_features",
    stage="feature_sets",
    version="1.0.0",
    schema_version="1.0",
    generated_by="example-pipeline",
    future_training_target="category_match"
)
print(f"Exported feature sets to: {csv_feat}")
print("Pipeline run complete. Ready for ML training in Batch 3!")